# Kidney Cancer Detection - Colab runner
Runtime > Change runtime type > **GPU (T4)** before starting. Run cells top to bottom.

## 1. Install

In [ ]:
!pip install -q pydicom scikit-image seaborn tcia_utils
import torch; print('GPU:', torch.cuda.is_available())

## 2. Upload the two scripts
Select `prepare_dataset.py` and `kidney_cancer_pipeline.py`.

In [ ]:
from google.colab import files
files.upload()

## 3. Smoke test (synthetic images - numbers are meaningless, only checks the code runs)

In [ ]:
!python kidney_cancer_pipeline.py --demo --out demo_results

## 4. Download a subset of C4KC-KiTS from TCIA
The full collection is 40.7 GB, so we take N patients (CT + SEG series). Increase N for a better project (40-60 is reasonable).

In [ ]:
import pandas as pd
from tcia_utils import nbia
N = 40
series = nbia.getSeries(collection="C4KC-KiTS")
df = pd.DataFrame(series)
print(df.Modality.value_counts())
pats = sorted(df.PatientID.unique())[:N]
sub = df[df.PatientID.isin(pats) & df.Modality.isin(["CT", "SEG"])]
print(len(pats), "patients,", len(sub), "series to download")
nbia.downloadSeries(sub.to_dict("records"), input_type="list", path="tciaDownload")

## 5. Convert DICOM + segmentations to labelled PNG slices

In [ ]:
!python prepare_dataset.py --src tciaDownload --out data --per-class 12

Check the printed **segment labels**. If `Tumor` / `Kidney` weren't recognised (0 tumour slices), rerun with `--tumor-regex` / `--kidney-regex` matching the names shown.

## 6. Train + evaluate (baselines vs proposed hybrid)

In [ ]:
!python kidney_cancer_pipeline.py --data data --out results --epochs 8

## 7. View results

In [ ]:
import pandas as pd
from IPython.display import Image, display
print(pd.read_csv('results/results.csv').to_string(index=False))
for f in ['roc','confusion_matrices','model_comparison','feature_importance','correlation','gradcam']:
    try: display(Image(f'results/{f}.png'))
    except Exception as e: print(f, 'missing')

## 8. Save everything (results + dataset summary) to your computer

In [ ]:
!zip -qr project_outputs.zip results dataset_summary.csv
files.download('project_outputs.zip')